# About This notebook

This is notebook imports BLS Wage datasets and prepares them for insertion into a postgresql database for further analysis. 

When completed, the idea is that this notebook is run once, transferring the data in as raw and pristine of a fashion as possible to the postgresql database.

If the data in the database is found to be unsatisfactory, the associated database tables would need to be dropped, this file would need to be adapted as necessary, and then run again. 

## Link to Data

[https://www.bls.gov/oes/](https://www.bls.gov/oes/)

# Project Setup

## Import and Verifications

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import sqlalchemy

In [2]:
print(sqlalchemy.__version__)

2.0.48


In [3]:
from sqlalchemy import create_engine
from sqlalchemy import text

## Create SQL Engine and Test Connection

In [4]:
engine = create_engine('postgresql+psycopg2://postgres@localhost/train_reward_compare')

In [5]:
with engine.connect() as conn:
    result = conn.execute(text("select count(*) from onet.occupation_data"))
    print(result.fetchone())

(1016,)


## Import BLS Files

In [6]:
file_nat = '../raw-data/bls-wage/oesm24nat/national_M2024_dl.xlsx'
file_sta = '../raw-data/bls-wage/oesm24st/state_M2024_dl.xlsx'
file_met = '../raw-data/bls-wage/oesm24ma/MSA_M2024_dl.xlsx'
file_bos = '../raw-data/bls-wage/oesm24ma/BOS_M2024_dl.xlsx'
file_nbs = '../raw-data/bls-wage/oesm24in4/natsector_M2024_dl.xlsx'

In [7]:
nat = pd.ExcelFile(file_nat)
sta = pd.ExcelFile(file_sta)
met = pd.ExcelFile(file_met)
bos = pd.ExcelFile(file_bos)
nbs = pd.ExcelFile(file_nbs)

## Test Data in Each Imported File

In [8]:
print(nat.sheet_names)
print(sta.sheet_names)
print(met.sheet_names)
print(bos.sheet_names)
print(nbs.sheet_names)

['national_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']
['state_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']
['MSA_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']
['BOS_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']
['Natsector_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']


# Rough Wrangling

## National Data

### Rough Wrangle National BLS Data

In [9]:
nat.parse('national_M2024_dl').head()

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,23.8,37.89,60.44,29990,36730,49500,78810,125720,NaN,NaN
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
4,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN


In [10]:
nat_data = nat.parse('national_M2024_dl')

In [11]:
nat_data.columns = nat_data.columns.str.lower()

In [12]:
nat_data.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,23.8,37.89,60.44,29990,36730,49500,78810,125720,NaN,NaN
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
4,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN


### Rough Wrangle National Field Descriptions

In [13]:
nat_desc_pre = nat.parse('Field Descriptions')

In [14]:
nat_desc_pre.iloc[:15,:]

,May 2024 OEWS Estimates,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Occupational Employment and Wage Statistics (O...,NaN,NaN
2,"Bureau of Labor Statistics, Department of Labor",NaN,NaN
3,website: www.bls.gov/oes,NaN,NaN
4,email: oewsinfo@bls.gov,NaN,NaN
5,NaN,NaN,NaN
6,Not all fields are available for every type of...,NaN,NaN
7,NaN,NaN,NaN
8,Field,Field Description,NaN
9,area,"U.S. (99), state FIPS code, Metropolitan Stati...",NaN


In [15]:
nat_desc_pre = nat_desc_pre.drop('Unnamed: 2', axis=1)

In [16]:
drop_index = nat_desc_pre.index[0:8]

In [17]:
nat_desc_pre_dropped = nat_desc_pre.drop(drop_index)

In [18]:
nat_desc_pre_dropped.head()

,May 2024 OEWS Estimates,Unnamed: 1
8,Field,Field Description
9,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
10,area_title,Area name
11,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
12,prim_state,"The primary state for the given area. ""US"" is ..."


In [19]:
nat_desc_pre_new_header = nat_desc_pre_dropped.iloc[0]

In [20]:
nat_desc_pre_new = nat_desc_pre_dropped[1:]

In [21]:
nat_desc_pre_new.columns = nat_desc_pre_new_header

In [22]:
nat_desc_pre_new = nat_desc_pre_new.reset_index(drop=True)

In [23]:
nat_desc_pre_new.index.name = None

In [24]:
nat_desc_pre_new = nat_desc_pre_new.rename_axis(None, axis=1)

In [25]:
nat_desc_pre_new.head()

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...


In [26]:
nat_desc_pre_new.tail(10)

,Field,Field Description
28,a_pct75,Annual 75th percentile wage
29,a_pct90,Annual 90th percentile wage
30,annual,"Contains ""TRUE"" if only annual wages are relea..."
31,hourly,"Contains ""TRUE"" if only hourly wages are relea..."
32,NaN,NaN
33,Notes:,NaN
34,* = indicates that a wage estimate is not ava...,NaN
35,** = indicates that an employment estimate is...,NaN
36,# = indicates a wage equal to or greater than...,NaN
37,~ =indicates that the percent of establishment...,NaN


#### Split Descriptions and Notes

In [27]:
notes_df = nat_desc_pre_new.iloc[32:,:]

In [28]:
nat_desc = nat_desc_pre_new.iloc[:32, :]

In [29]:
nat_desc

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...
5,naics_title,North American Industry Classification System ...
6,i_group,Industry level. Indicates cross-industry or NA...
7,own_code,Ownership type: 1= Federal Government; 2= Stat...
8,occ_code,The 6-digit Standard Occupational Classificati...
9,occ_title,SOC title or OEWS-specific title for the occup...


## Storing Notes in DataFrame

In [30]:
notes_df = notes_df.drop(columns=['Field Description'], axis=1)

In [31]:
notes_df = notes_df.iloc[2:,:]

In [32]:
notes_df = notes_df.rename(columns={'Field':'Notes'}) 

In [33]:
notes_df

,Notes
34,* = indicates that a wage estimate is not ava...
35,** = indicates that an employment estimate is...
36,# = indicates a wage equal to or greater than...
37,~ =indicates that the percent of establishment...


## State Data

### Rough Wrangle State Data

In [34]:
sta.parse('state_M2024_dl').head()

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,21.07,30.82,47.51,23520,30660,43830,64110,98810,NaN,NaN
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,48.39,68.5,98.03,51100,72870,100640,142480,203900,NaN,NaN
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,79.04,106.69,#,104950,130950,164400,221910,#,NaN,NaN
3,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,51.12,78.26,#,50410,74720,106330,162780,#,NaN,NaN
4,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,*,*,18270,20950,26990,41760,63900,True,NaN


In [35]:
sta_data = sta.parse('state_M2024_dl')

In [36]:
sta_data.columns = sta_data.columns.str.lower()

In [37]:
sta_data.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,21.07,30.82,47.51,23520,30660,43830,64110,98810,NaN,NaN
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,48.39,68.5,98.03,51100,72870,100640,142480,203900,NaN,NaN
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,79.04,106.69,#,104950,130950,164400,221910,#,NaN,NaN
3,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,51.12,78.26,#,50410,74720,106330,162780,#,NaN,NaN
4,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,*,*,18270,20950,26990,41760,63900,True,NaN


### Rough Wrangle State Field Descriptions

In [38]:
sta_desc = sta.parse('Field Descriptions')

In [39]:
sta_desc.iloc[:15,:]

,May 2024 OEWS Estimates,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Occupational Employment and Wage Statistics (O...,NaN,NaN
2,"Bureau of Labor Statistics, Department of Labor",NaN,NaN
3,website: www.bls.gov/oes,NaN,NaN
4,email: oewsinfo@bls.gov,NaN,NaN
5,NaN,NaN,NaN
6,Not all fields are available for every type of...,NaN,NaN
7,NaN,NaN,NaN
8,Field,Field Description,NaN
9,area,"U.S. (99), state FIPS code, Metropolitan Stati...",NaN


In [40]:
sta_desc = sta_desc.drop('Unnamed: 2', axis=1)
drop_index = sta_desc.index[0:8]
sta_desc_dropped = sta_desc.drop(drop_index)
sta_desc_new_header = sta_desc_dropped.iloc[0]
sta_desc_new = sta_desc_dropped[1:]
sta_desc_new.columns = sta_desc_new_header
sta_desc_new = sta_desc_new.reset_index(drop=True)
sta_desc_new.index.name = None
sta_desc_new = sta_desc_new.rename_axis(None, axis=1)

In [41]:
sta_desc_new.head()

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...


## Metropolitan Data

### Rough Wrangle Metropolitan Data

In [42]:
met_data = met.parse('MSA_M2024_dl')
met_data.columns = met_data.columns.str.lower()

In [43]:
met_data.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,20.25,29.34,45.3,23120,29640,42120,61020,94220,NaN,NaN
1,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,42.03,61.17,84.3,44710,61250,87420,127230,175350,NaN,NaN
2,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,85.35,#,#,99140,107040,177520,#,#,NaN,NaN
3,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,37.18,58.64,87.22,35980,50640,77340,121970,181420,NaN,NaN
4,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-2021,Marketing Managers,...,50.02,73.86,99.9,58480,82560,104030,153630,207790,NaN,NaN


### Rough Wrangle Metropolitan Field Descriptions

In [44]:
met_desc = met.parse('Field Descriptions')

In [45]:
met_desc.iloc[:15,:]

,May 2024 OEWS Estimates,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Occupational Employment and Wage Statistics (O...,NaN,NaN
2,"Bureau of Labor Statistics, Department of Labor",NaN,NaN
3,website: www.bls.gov/oes,NaN,NaN
4,email: oewsinfo@bls.gov,NaN,NaN
5,NaN,NaN,NaN
6,Not all fields are available for every type of...,NaN,NaN
7,NaN,NaN,NaN
8,Field,Field Description,NaN
9,area,"U.S. (99), state FIPS code, Metropolitan Stati...",NaN


In [46]:
met_desc = met_desc.drop('Unnamed: 2', axis=1)
drop_index = met_desc.index[0:8]
met_desc_dropped = met_desc.drop(drop_index)
met_desc_new_header = met_desc_dropped.iloc[0]
met_desc_new = met_desc_dropped[1:]
met_desc_new.columns = met_desc_new_header
met_desc_new = met_desc_new.reset_index(drop=True)
met_desc_new.index.name = None
met_desc_new = met_desc_new.rename_axis(None, axis=1)

In [47]:
met_desc_new.head()

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...


## Balance of State Data

### Rough Wrangle Balance of State Data

In [48]:
bos_data = bos.parse('BOS_M2024_dl')
bos_data.columns = bos_data.columns.str.lower() 

In [49]:
bos_data.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,18.94,25,35.51,22380,29690,39400,52000,73870,NaN,NaN
1,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,40.54,58.5,80.72,44770,60730,84320,121680,167910,NaN,NaN
2,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,71.58,85.16,#,114610,121930,148890,177130,#,NaN,NaN
3,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,46.63,73.99,101.78,42680,63380,96980,153900,211700,NaN,NaN
4,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,*,*,18210,22690,51460,52020,52020,True,NaN


### Rough Wrangle Balance of State Field Descriptions

In [50]:
bos_desc = bos.parse('Field Descriptions')

In [51]:
bos_desc.iloc[:15,:]

,May 2024 OEWS Estimates,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Occupational Employment and Wage Statistics (O...,NaN,NaN
2,"Bureau of Labor Statistics, Department of Labor",NaN,NaN
3,website: www.bls.gov/oes,NaN,NaN
4,email: oewsinfo@bls.gov,NaN,NaN
5,NaN,NaN,NaN
6,Not all fields are available for every type of...,NaN,NaN
7,NaN,NaN,NaN
8,Field,Field Description,NaN
9,area,"U.S. (99), state FIPS code, Metropolitan Stati...",NaN


In [52]:
bos_desc = bos_desc.drop('Unnamed: 2', axis=1)
drop_index = bos_desc.index[0:8]
bos_desc_dropped = bos_desc.drop(drop_index)
bos_desc_new_header = bos_desc_dropped.iloc[0]
bos_desc_new = bos_desc_dropped[1:]
bos_desc_new.columns = bos_desc_new_header
bos_desc_new = bos_desc_new.reset_index(drop=True)
bos_desc_new.index.name = None
bos_desc_new = bos_desc_new.rename_axis(None, axis=1)

In [53]:
bos_desc_new.head()

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...


## National Sector Data

### Rough Wrangle National Sector Data

In [54]:
nbs_data = nbs.parse('Natsector_M2024_dl')
nbs_data.columns = nbs_data.columns.str.lower() 

In [55]:
nbs_data.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,99,U.S.,1,US,11,"Agriculture, Forestry, Fishing and Hunting",sector,5,00-0000,All Occupations,...,17.66,22.21,30.23,33280,34680,36720,46200,62880,NaN,NaN
1,99,U.S.,1,US,11,"Agriculture, Forestry, Fishing and Hunting",sector,5,11-0000,Management Occupations,...,46.08,63.4,90.74,41710,67090,95840,131870,188730,NaN,NaN
2,99,U.S.,1,US,11,"Agriculture, Forestry, Fishing and Hunting",sector,5,11-1000,Top Executives,...,40.91,61.6,91.98,32100,57390,85080,128140,191320,NaN,NaN
3,99,U.S.,1,US,11,"Agriculture, Forestry, Fishing and Hunting",sector,5,11-1010,Chief Executives,...,103.99,#,#,105020,154280,216290,#,#,NaN,NaN
4,99,U.S.,1,US,11,"Agriculture, Forestry, Fishing and Hunting",sector,5,11-1011,Chief Executives,...,103.99,#,#,105020,154280,216290,#,#,NaN,NaN


### Rough Wrangle National Sector Field Descriptions

In [56]:
nbs_desc = nbs.parse('Field Descriptions')

In [57]:
nbs_desc.iloc[:15,:]

,May 2024 OEWS Estimates,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,Occupational Employment and Wage Statistics (O...,NaN,NaN
2,"Bureau of Labor Statistics, Department of Labor",NaN,NaN
3,website: www.bls.gov/oes,NaN,NaN
4,email: oewsinfo@bls.gov,NaN,NaN
5,NaN,NaN,NaN
6,Not all fields are available for every type of...,NaN,NaN
7,NaN,NaN,NaN
8,Field,Field Description,NaN
9,area,"U.S. (99), state FIPS code, Metropolitan Stati...",NaN


In [58]:
nbs_desc = nbs_desc.drop('Unnamed: 2', axis=1)
drop_index = nbs_desc.index[0:8]
nbs_desc_dropped = nbs_desc.drop(drop_index)
nbs_desc_new_header = nbs_desc_dropped.iloc[0]
nbs_desc_new = nbs_desc_dropped[1:]
nbs_desc_new.columns = nbs_desc_new_header
nbs_desc_new = nbs_desc_new.reset_index(drop=True)
nbs_desc_new.index.name = None
nbs_desc_new = nbs_desc_new.rename_axis(None, axis=1)

In [59]:
nbs_desc_new.head()

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...


## Rough Wrangle Closing Commentary

The `Field Description` dataframes are ready to be pushed to the sql database. I do want them readily available as a part of the long-term project, so that either I or my users can reference them.

The data for each of the five sets, on the other hand, is a mixed bag. I'm seeing `NaN` values and `#` values. I do not know what they mean. It's time to start investigating there.

# Full Wrangle

## National Data

### Basic Inspection

In [60]:
nat_data.dtypes

area              int64
area_title       object
area_type         int64
prim_state       object
naics             int64
naics_title      object
i_group          object
own_code          int64
occ_code         object
occ_title        object
o_group          object
tot_emp           int64
emp_prse        float64
jobs_1000       float64
loc_quotient    float64
pct_total       float64
pct_rpt         float64
h_mean           object
a_mean           object
mean_prse       float64
h_pct10          object
h_pct25          object
h_median         object
h_pct75          object
h_pct90          object
a_pct10          object
a_pct25          object
a_median         object
a_pct75          object
a_pct90          object
annual           object
hourly           object
dtype: object

In [61]:
nat_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1403 entries, 0 to 1402
Data columns (total 32 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area          1403 non-null   int64  
 1   area_title    1403 non-null   object 
 2   area_type     1403 non-null   int64  
 3   prim_state    1403 non-null   object 
 4   naics         1403 non-null   int64  
 5   naics_title   1403 non-null   object 
 6   i_group       1403 non-null   object 
 7   own_code      1403 non-null   int64  
 8   occ_code      1403 non-null   object 
 9   occ_title     1403 non-null   object 
 10  o_group       1403 non-null   object 
 11  tot_emp       1403 non-null   int64  
 12  emp_prse      1403 non-null   float64
 13  jobs_1000     0 non-null      float64
 14  loc_quotient  0 non-null      float64
 15  pct_total     0 non-null      float64
 16  pct_rpt       0 non-null      float64
 17  h_mean        1403 non-null   object 
 18  a_mean        1403 non-null 

### Removing Blank Columns

We see that there are four entirely blank columns.

Let's check their description.

In [62]:
with pd.option_context('display.max_colwidth', None):
    print(nat_desc.iloc[13:17,:])

           Field  \
13     jobs_1000   
14  loc quotient   
15     pct_total   
16       pct_rpt   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      Field Description  
13                                                                                                                                                                                                                                                                                                                          The number of jobs (employment) in the given occupation per 1,000 jobs in the given area. Onl

#### Confirm Necessity of Deletion

These four columns can be dropped from the `nat_data` df. 

#### Add To Do List

##### Data
- Drop the four columns that have all blanks

##### Descriptions
- Drop the rows to match the dropped columns in the data

##### Remote Columns from National Data

In [63]:
drop_nat_data_list = ['jobs_1000', 'loc_quotient', 'pct_total', 'pct_rpt']

In [64]:
nat_data = nat_data.drop(drop_nat_data_list, axis=1)

In [65]:
nat_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1403 entries, 0 to 1402
Data columns (total 28 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   area         1403 non-null   int64  
 1   area_title   1403 non-null   object 
 2   area_type    1403 non-null   int64  
 3   prim_state   1403 non-null   object 
 4   naics        1403 non-null   int64  
 5   naics_title  1403 non-null   object 
 6   i_group      1403 non-null   object 
 7   own_code     1403 non-null   int64  
 8   occ_code     1403 non-null   object 
 9   occ_title    1403 non-null   object 
 10  o_group      1403 non-null   object 
 11  tot_emp      1403 non-null   int64  
 12  emp_prse     1403 non-null   float64
 13  h_mean       1403 non-null   object 
 14  a_mean       1403 non-null   object 
 15  mean_prse    1403 non-null   float64
 16  h_pct10      1403 non-null   object 
 17  h_pct25      1403 non-null   object 
 18  h_median     1403 non-null   object 
 19  h_pct7

##### Remote Rows from National Descriptions

In [66]:
nat_desc = nat_desc[nat_desc['Field'].isin(drop_nat_data_list) == False]

In [67]:
nat_desc

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...
5,naics_title,North American Industry Classification System ...
6,i_group,Industry level. Indicates cross-industry or NA...
7,own_code,Ownership type: 1= Federal Government; 2= Stat...
8,occ_code,The 6-digit Standard Occupational Classificati...
9,occ_title,SOC title or OEWS-specific title for the occup...


#### Cross Off To Do List

##### Data
- ~~Drop the four columns that have all blanks~~

##### Descriptions
- ~~Drop the rows to match the dropped columns in the data~~

### National Data Area-Related Columns Wrangling

Dealing with the first four rows, `area`, `area_title`, `area_type`, `prim_state`.

The first and third row are of type `int`, while the second and fourths are`object` types, suggesting either strings or even multiple types.

All values are non-null.

We need to make sure that the first and third do not contain values outside the scope outline in the description.

We need to inspect the second and fourth for consistency as well.

In [68]:
nat_data['area'].describe()

count    1403.0
mean       99.0
std         0.0
min        99.0
25%        99.0
50%        99.0
75%        99.0
max        99.0
Name: area, dtype: float64

That's interesting. Apparently, the entire first row is essentially meaningless.

In [69]:
print(nat_desc.loc[nat_desc['Field'] == 'area', 'Field Description'].values[0])

U.S. (99), state FIPS code, Metropolitan Statistical Area (MSA) code, or OEWS-specific nonmetropolitan area code 


This appears to be a column that only has variance in other datasets. The value of this column in this context is limited, but it may be useful elsewhere. Therefore, we will keep it as is, but perhaps discard it when we begin inference.

In [70]:
nat_data['area_title'].head()

0    U.S.
1    U.S.
2    U.S.
3    U.S.
4    U.S.
Name: area_title, dtype: object

In [71]:
nat_data['area_title'].unique()

array(['U.S.'], dtype=object)

The lack of variance in the `area_title` column follows.

In [72]:
nat_data['area_type'].unique()

array([1])

The results for `area_type` are identical.

In [73]:
nat_data['prim_state'].unique()

array(['US'], dtype=object)

The result of this study is that these columns all simply indicate that they are exactly what the title of the dataset would imply: National.

We will keep these for the PostgreSQL insertion at this time.

### Industry Classifications

Looking at `naics`, `naics_title`, and `i_group`.

In [74]:
nat_data['naics'].unique()

array([0])

In [75]:
print(nat_desc.loc[nat_desc['Field'] == 'naics', 'Field Description'].values[0])

North American Industry Classification System (NAICS) code for the given industry 


In [76]:
nat_data['naics_title'].unique()

array(['Cross-industry'], dtype=object)

In [77]:
print(nat_desc.loc[nat_desc['Field'] == 'naics_title', 'Field Description'].values[0])

North American Industry Classification System (NAICS) title for the given industry 


In [78]:
nat_data['i_group'].unique()

array(['cross-industry'], dtype=object)

In [79]:
print(nat_desc.loc[nat_desc['Field'] == 'i_group', 'Field Description'].values[0])

Industry level. Indicates cross-industry or NAICS sector, 3-digit, 4-digit, 5-digit, or 6-digit industry. For industries that OEWS no longer publishes at the 4-digit NAICS level, the “4-digit” designation indicates the most detailed industry breakdown available: either a standard NAICS 3-digit industry or an OEWS-specific combination of 4-digit industries. Industries that OEWS has aggregated to the 3-digit NAICS level (for example, NAICS 327000) will appear twice, once with the “3-digit” and once with the “4-digit” designation. 


In [80]:
nat_data['own_code'].unique()	

array([1235])

In [81]:
print(nat_desc.loc[nat_desc['Field'] == 'own_code', 'Field Description'].values[0])

Ownership type: 1= Federal Government; 2= State Government; 3= Local Government; 123= Federal, State, and Local Government; 235=Private, State, and Local Government; 35 = Private and Local Government; 5= Private; 57=Private, Local Government Gambling Establishments (Sector 71), and Local Government Casino Hotels (Sector 72); 58= Private plus State and Local Government Hospitals; 59= Private and Postal Service; 1235= Federal, State, and Local Government and Private Sector 


These columns follow the same pattern as before in terms of variance.

We will keep the values for the PostgreSQL database, even though they will not have value in isolation from other datasets.

### `occ_code`: Standard Occupation Classifications

In [82]:
nat_data['occ_code'].unique()

array(['00-0000', '11-0000', '11-1000', ..., '53-7121', '53-7190',
       '53-7199'], dtype=object)

In [83]:
print(nat_desc.loc[nat_desc['Field'] == 'occ_code', 'Field Description'].values[0])

The 6-digit Standard Occupational Classification (SOC) code or OEWS-specific code for the occupation 


In [84]:
nat_data['occ_code'].describe()

count        1403
unique       1396
top       31-1120
freq            2
Name: occ_code, dtype: object

In [85]:
nat_data['occ_code'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 1403 entries, 0 to 1402
Series name: occ_code
Non-Null Count  Dtype 
--------------  ----- 
1403 non-null   object
dtypes: object(1)
memory usage: 11.1+ KB


In [86]:
print(type(nat_data['occ_code'][0]))

<class 'str'>


In [87]:
counts = nat_data['occ_code'].value_counts()

In [88]:
counts

occ_code
31-1120    2
47-4090    2
29-2010    2
51-2090    2
39-7010    2
          ..
27-2012    1
27-2011    1
27-2010    1
27-2000    1
53-7199    1
Name: count, Length: 1396, dtype: int64

In [89]:
threshold = 1

In [90]:
counts_thresh = counts[counts > threshold].index

In [91]:
counts_thresh

Index(['31-1120', '47-4090', '29-2010', '51-2090', '39-7010', '13-1020',
       '13-2020'],
      dtype='object', name='occ_code')

It seems that there are a few areas that are duplicated.

In [92]:
nat_data[nat_data.groupby('occ_code')['occ_code'].transform('count') > threshold]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
78,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,13-1020,Buyers and Purchasing Agents,...,36.37,47.69,61.31,46460,58670,75650,99190,127520,NaN,NaN
79,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,13-1020,Buyers and Purchasing Agents,...,36.37,47.69,61.31,46460,58670,75650,99190,127520,NaN,NaN
111,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,13-2020,Property Appraisers and Assessors,...,31.45,43.66,59.02,38480,49310,65420,90810,122760,NaN,NaN
112,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,13-2020,Property Appraisers and Assessors,...,31.45,43.66,59.02,38480,49310,65420,90810,122760,NaN,NaN
573,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,29-2010,Clinical Laboratory Technologists and Technicians,...,29.75,38.46,47.11,38020,46580,61890,80010,97990,NaN,NaN
574,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,29-2010,Clinical Laboratory Technologists and Technicians,...,29.75,38.46,47.11,38020,46580,61890,80010,97990,NaN,NaN
612,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,31-1120,Home Health and Personal Care Aides,...,16.78,18.26,21.25,25600,30370,34900,37980,44190,NaN,NaN
613,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,31-1120,Home Health and Personal Care Aides,...,16.78,18.26,21.25,25600,30370,34900,37980,44190,NaN,NaN
779,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-7010,Tour and Travel Guides,...,17.63,22.07,28.81,26890,31250,36660,45910,59930,NaN,NaN
780,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-7010,Tour and Travel Guides,...,17.63,22.07,28.81,26890,31250,36660,45910,59930,NaN,NaN


We appear to have found fourteen duplicate rows. These must be cleaned.

#### Add To Do Item

- Clean data duplicates

Before cleaning them, let's take a moment to understand by looking at the [SOC User Guide](https://www.bls.gov/soc/socguide.htm#Ques3).

> The 2000 SOC classifies workers at four levels of aggregation: 1) major group; 2) minor group; 3) broad occupation; and 4) detailed occupation. All occupations are clustered into one of the following 23 major groups:

>  Within these major groups are 96 minor groups, 449 broad occupations, and 821 detailed occupations. Occupations with similar skills or work activities are grouped at each of the four levels of hierarchy to facilitate comparisons. For example, "Life, Physical and Social Science Occupations" (19-0000) is divided into four minor groups, "Life Scientists" (19-1000), "Physical Scientists" (19-2000), "Social Scientists and Related Workers" (19-3000), and "Life, Physical and Social Science Technicians" (19-4000). Life Scientists contains broad occupations such as "Agriculture and Food Scientists" (19-1010), and "Biological Scientists" (19-1020). The broad occupation Biological Scientists includes detailed occupations such as "Biochemists and Biophysicists" (19-1021), and "Microbiologists" (19-1022).

How many `o_group` values do we have before we begin cleaning?

In [93]:
nat_data['o_group'].value_counts()

o_group
detailed    831
broad       455
minor        94
major        22
total         1
Name: count, dtype: int64

We appear to have less of each group than the number reported in the user guide. 

Since this dataset has not yet dropped any rows, we can assume that the dataset originated in this form.

#### Clean Duplicate Rows

In [94]:
nat_data = nat_data.drop_duplicates()

In [95]:
counts = nat_data['occ_code'].value_counts()

In [96]:
counts

occ_code
31-1120    2
47-4090    2
29-2010    2
51-2090    2
39-7010    2
          ..
27-2012    1
27-2011    1
27-2010    1
27-2000    1
53-7199    1
Name: count, Length: 1396, dtype: int64

They don't seem to be dropped. Perhaps there are hidden rows that are different?

In [97]:
with pd.option_context('display.max_columns', None):
    print(nat_data[nat_data.groupby('occ_code')['occ_code'].transform('count') > threshold])

      area area_title  area_type prim_state  naics     naics_title  \
78      99       U.S.          1         US      0  Cross-industry   
79      99       U.S.          1         US      0  Cross-industry   
111     99       U.S.          1         US      0  Cross-industry   
112     99       U.S.          1         US      0  Cross-industry   
573     99       U.S.          1         US      0  Cross-industry   
574     99       U.S.          1         US      0  Cross-industry   
612     99       U.S.          1         US      0  Cross-industry   
613     99       U.S.          1         US      0  Cross-industry   
779     99       U.S.          1         US      0  Cross-industry   
780     99       U.S.          1         US      0  Cross-industry   
1045    99       U.S.          1         US      0  Cross-industry   
1046    99       U.S.          1         US      0  Cross-industry   
1163    99       U.S.          1         US      0  Cross-industry   
1164    99       U.S

This shows us that the `o_group` column is providing some type of difference in these columns. Let's explore this column's description.

In [98]:
print(nat_desc.loc[nat_desc['Field'] == 'o_group', 'Field Description'].values[0])

SOC occupation level. For most occupations, this field indicates the standard SOC major, minor, broad, and detailed levels, in addition to all-occupations totals. For occupations that OEWS no longer publishes at the SOC detailed level, the “detailed” designation indicates the most detailed data available: either a standard SOC broad occupation or an OEWS-specific combination of detailed occupations. Occupations that OEWS has aggregated to the SOC broad occupation level will appear in the file twice, once with the “broad” and once with the “detailed” designation.


This informs us that these are rows of data that have been included on purpose twice, due to changes in the way Occupational Employment and Wage Statistics (OEWS) manages the data.

The duplicates that have the `detailed` values can be dropped, as OEWS has aggregated them into the `broad` category.

In [99]:
nat_data_filtered = nat_data[(nat_data['occ_code'].isin(counts_thresh)) & (nat_data['o_group'] == 'detailed')]

In [100]:
nat_data_filtered

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
79,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,13-1020,Buyers and Purchasing Agents,...,36.37,47.69,61.31,46460,58670,75650,99190,127520,NaN,NaN
112,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,13-2020,Property Appraisers and Assessors,...,31.45,43.66,59.02,38480,49310,65420,90810,122760,NaN,NaN
574,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,29-2010,Clinical Laboratory Technologists and Technicians,...,29.75,38.46,47.11,38020,46580,61890,80010,97990,NaN,NaN
613,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,31-1120,Home Health and Personal Care Aides,...,16.78,18.26,21.25,25600,30370,34900,37980,44190,NaN,NaN
780,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-7010,Tour and Travel Guides,...,17.63,22.07,28.81,26890,31250,36660,45910,59930,NaN,NaN
1046,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,47-4090,Miscellaneous Construction and Related Workers,...,23.14,29.31,37.28,35160,39990,48120,60960,77540,NaN,NaN
1164,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,51-2090,Miscellaneous Assemblers and Fabricators,...,20.29,23.76,29.82,31650,36660,42210,49410,62030,NaN,NaN


In [101]:
nat_data_filtered['o_group']

79      detailed
112     detailed
574     detailed
613     detailed
780     detailed
1046    detailed
1164    detailed
Name: o_group, dtype: object

In [102]:
nat_data_pre_drop = nat_data.drop(nat_data_filtered.index)

In [103]:
nat_data_pre_drop.head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,23.8,37.89,60.44,29990,36730,49500,78810,125720,NaN,NaN
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
4,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN


In [104]:
nat_data_pre_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1396 entries, 0 to 1402
Data columns (total 28 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   area         1396 non-null   int64  
 1   area_title   1396 non-null   object 
 2   area_type    1396 non-null   int64  
 3   prim_state   1396 non-null   object 
 4   naics        1396 non-null   int64  
 5   naics_title  1396 non-null   object 
 6   i_group      1396 non-null   object 
 7   own_code     1396 non-null   int64  
 8   occ_code     1396 non-null   object 
 9   occ_title    1396 non-null   object 
 10  o_group      1396 non-null   object 
 11  tot_emp      1396 non-null   int64  
 12  emp_prse     1396 non-null   float64
 13  h_mean       1396 non-null   object 
 14  a_mean       1396 non-null   object 
 15  mean_prse    1396 non-null   float64
 16  h_pct10      1396 non-null   object 
 17  h_pct25      1396 non-null   object 
 18  h_median     1396 non-null   object 
 19  h_pct75    

We see that seven rows have been removed.

In [105]:
nat_data_pre_drop[nat_data_pre_drop.groupby('occ_code')['occ_code'].transform('count') > threshold]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly


There are no duplicates.

In [106]:
nat_data = nat_data_pre_drop

#### Mark To Do Item Complete

- ~~Clean data duplicates~~

### `occ_title`: SOC Column Cleaning Continuation

In [107]:
nat_data['occ_title'].unique()

array(['All Occupations', 'Management Occupations', 'Top Executives', ...,
       'Tank Car, Truck, and Ship Loaders',
       'Miscellaneous Material Moving Workers',
       'Material Moving Workers, All Other'], dtype=object)

In [108]:
print(nat_desc.loc[nat_desc['Field'] == 'occ_title', 'Field Description'].values[0])

SOC title or OEWS-specific title for the occupation


In [109]:
nat_data['occ_title'].describe()

count                   1396
unique                  1138
top       Massage Therapists
freq                       2
Name: occ_title, dtype: object

In [110]:
nat_data['occ_title'].info()

<class 'pandas.core.series.Series'>
Index: 1396 entries, 0 to 1402
Series name: occ_title
Non-Null Count  Dtype 
--------------  ----- 
1396 non-null   object
dtypes: object(1)
memory usage: 21.8+ KB


In [111]:
print(type(nat_data['occ_title'][0]))

<class 'str'>


In [112]:
counts = nat_data['occ_title'].value_counts()

In [113]:
counts

occ_title
Massage Therapists                                 2
Biological Technicians                             2
Couriers and Messengers                            2
Cargo and Freight Agents                           2
Printing Workers                                   2
                                                  ..
Healthcare Diagnosing or Treating Practitioners    1
Dentists                                           1
Dentists, General                                  1
Oral and Maxillofacial Surgeons                    1
Material Moving Workers, All Other                 1
Name: count, Length: 1138, dtype: int64

We seem to have found more duplicate values.

In [114]:
nat_data[nat_data.groupby('occ_title')['occ_title'].transform('count') > threshold]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
4,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
5,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1020,General and Operations Managers,...,49.5,78.91,#,47420,67160,102950,164130,#,NaN,NaN
6,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,49.5,78.91,#,47420,67160,102950,164130,#,NaN,NaN
7,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1030,Legislators,...,*,*,*,20380,29120,44810,80350,137820,True,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1386,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7051,Industrial Truck and Tractor Operators,...,22.3,25.81,29.59,36500,39780,46390,53680,61540,NaN,NaN
1397,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7080,Refuse and Recyclable Material Collectors,...,23.24,29.33,36.16,31810,38330,48350,61010,75200,NaN,NaN
1398,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7081,Refuse and Recyclable Material Collectors,...,23.24,29.33,36.16,31810,38330,48350,61010,75200,NaN,NaN
1399,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7120,"Tank Car, Truck, and Ship Loaders",...,27.92,34.25,42.36,38260,47260,58070,71230,88120,NaN,NaN


In [115]:
with pd.option_context('display.max_columns', None):
    print(nat_data[nat_data.groupby('occ_title')['occ_title'].transform('count') > threshold].head(2))

   area area_title  area_type prim_state  naics     naics_title  \
3    99       U.S.          1         US      0  Cross-industry   
4    99       U.S.          1         US      0  Cross-industry   

          i_group  own_code occ_code         occ_title   o_group  tot_emp  \
3  cross-industry      1235  11-1010  Chief Executives     broad   211850   
4  cross-industry      1235  11-1011  Chief Executives  detailed   211850   

   emp_prse  h_mean  a_mean  mean_prse h_pct10 h_pct25 h_median h_pct75  \
3       1.2  126.41  262930        0.9   35.44   60.61    99.24       #   
4       1.2  126.41  262930        0.9   35.44   60.61    99.24       #   

  h_pct90 a_pct10 a_pct25 a_median a_pct75 a_pct90 annual hourly  
3       #   73710  126080   206420       #       #    NaN    NaN  
4       #   73710  126080   206420       #       #    NaN    NaN  


These duplicates seem to have a different value in the `occ_code` and the `o_group` code. 

Likely, the `o_group` issue is a continuation of the issue from before, where a formerly `detailed` dataset has been aggregated into a `broad` dataset.

The `occ_code` appears to be raised by one digit lower as a part of the aggregation methodology shift.

Let's consider that the `detailed` rows can, again, are duplicates and can therefore be dropped. 

#### Add To Do Item

- Remove duplicate rows discovered via the `occ_title` column.

#### Clean Duplicate Rows

In [116]:
counts_thresh = counts[counts > threshold].index

In [117]:
nat_data_filtered = nat_data[(nat_data['occ_title'].isin(counts_thresh)) & (nat_data['o_group'] == 'detailed')]

In [118]:
nat_data_filtered

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
4,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
6,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,49.5,78.91,#,47420,67160,102950,164130,#,NaN,NaN
8,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,*,*,20380,29120,44810,80350,137820,True,NaN
11,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-2011,Advertising and Promotions Managers,...,61.04,85.85,#,63000,85990,126960,178570,#,NaN,NaN
23,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-3021,Computer and Information Systems Managers,...,82.31,103.95,#,104450,134350,171200,216220,#,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1382,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7031,Dredge Operators,...,23.28,28.99,36.08,42060,46120,48430,60300,75050,NaN,NaN
1384,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7041,Hoist and Winch Operators,...,25.15,43.37,55.83,33910,39220,52310,90200,116120,NaN,NaN
1386,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7051,Industrial Truck and Tractor Operators,...,22.3,25.81,29.59,36500,39780,46390,53680,61540,NaN,NaN
1398,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7081,Refuse and Recyclable Material Collectors,...,23.24,29.33,36.16,31810,38330,48350,61010,75200,NaN,NaN


In [119]:
nat_data_filtered['o_group'].unique()

array(['detailed'], dtype=object)

On the one hand, we seem to have made some progress. We have found `249` rows that can be dropped. 

However, there were `516` rows originally. We are missing perhaps `9` rows. 

While we could write some complicated logic at this stage to try to figure out why those rows are not yet included, we don't have to do so. When we drop the known duplicate rows and rerun the original duplicate-finding logic, we'll see them in isolation.

In [120]:
nat_data_pre_drop = nat_data.drop(nat_data_filtered.index)

In [121]:
nat_data_pre_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1147 entries, 0 to 1402
Data columns (total 28 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   area         1147 non-null   int64  
 1   area_title   1147 non-null   object 
 2   area_type    1147 non-null   int64  
 3   prim_state   1147 non-null   object 
 4   naics        1147 non-null   int64  
 5   naics_title  1147 non-null   object 
 6   i_group      1147 non-null   object 
 7   own_code     1147 non-null   int64  
 8   occ_code     1147 non-null   object 
 9   occ_title    1147 non-null   object 
 10  o_group      1147 non-null   object 
 11  tot_emp      1147 non-null   int64  
 12  emp_prse     1147 non-null   float64
 13  h_mean       1147 non-null   object 
 14  a_mean       1147 non-null   object 
 15  mean_prse    1147 non-null   float64
 16  h_pct10      1147 non-null   object 
 17  h_pct25      1147 non-null   object 
 18  h_median     1147 non-null   object 
 19  h_pct75    

#### Part 2: Analyzing for Additional Duplicates via `occ_title`

With the first set of duplicates removed, we should now be able to see the remaining duplicates.

In [122]:
counts = nat_data_pre_drop['occ_title'].value_counts()

In [123]:
counts

occ_title
Supervisors of Food Preparation and Serving Workers    2
Baggage Porters, Bellhops, and Concierges              2
Sales Representatives, Wholesale and Manufacturing     2
Grounds Maintenance Workers                            2
Tour and Travel Guides                                 2
                                                      ..
Athletes, Coaches, Umpires, and Related Workers        1
Athletes and Sports Competitors                        1
Coaches and Scouts                                     1
Umpires, Referees, and Other Sports Officials          1
Material Moving Workers, All Other                     1
Name: count, Length: 1138, dtype: int64

In [124]:
nat_data_pre_drop[nat_data_pre_drop.groupby('occ_title')['occ_title'].transform('count') > threshold]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
304,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,19-5000,Occupational Health and Safety Specialists and...,...,37.93,48.79,61.13,47790,59620,78900,101490,127140,NaN,NaN
305,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,19-5010,Occupational Health and Safety Specialists and...,...,37.93,48.79,61.13,47790,59620,78900,101490,127140,NaN,NaN
681,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,35-1000,Supervisors of Food Preparation and Serving Wo...,...,21.22,27.4,34.58,29600,35840,44140,56990,71920,NaN,NaN
682,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,35-1010,Supervisors of Food Preparation and Serving Wo...,...,21.22,27.4,34.58,29600,35840,44140,56990,71920,NaN,NaN
725,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,37-3000,Grounds Maintenance Workers,...,18.5,22.3,27.14,30140,35470,38470,46390,56460,NaN,NaN
726,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,37-3010,Grounds Maintenance Workers,...,18.5,22.3,27.14,30140,35470,38470,46390,56460,NaN,NaN
774,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-6000,"Baggage Porters, Bellhops, and Concierges",...,17.7,20.96,26.13,28360,32560,36810,43600,54340,NaN,NaN
775,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-6010,"Baggage Porters, Bellhops, and Concierges",...,17.7,20.96,26.13,28360,32560,36810,43600,54340,NaN,NaN
778,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-7000,Tour and Travel Guides,...,17.63,22.07,28.81,26890,31250,36660,45910,59930,NaN,NaN
779,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-7010,Tour and Travel Guides,...,17.63,22.07,28.81,26890,31250,36660,45910,59930,NaN,NaN


There are still `9` duplicates, as expected.

In [125]:
with pd.option_context('display.max_columns', None):
    print(nat_data_pre_drop[nat_data_pre_drop.groupby('occ_title')['occ_title'].transform('count') > threshold])

      area area_title  area_type prim_state  naics     naics_title  \
304     99       U.S.          1         US      0  Cross-industry   
305     99       U.S.          1         US      0  Cross-industry   
681     99       U.S.          1         US      0  Cross-industry   
682     99       U.S.          1         US      0  Cross-industry   
725     99       U.S.          1         US      0  Cross-industry   
726     99       U.S.          1         US      0  Cross-industry   
774     99       U.S.          1         US      0  Cross-industry   
775     99       U.S.          1         US      0  Cross-industry   
778     99       U.S.          1         US      0  Cross-industry   
779     99       U.S.          1         US      0  Cross-industry   
816     99       U.S.          1         US      0  Cross-industry   
817     99       U.S.          1         US      0  Cross-industry   
917     99       U.S.          1         US      0  Cross-industry   
918     99       U.S

This shows that these duplicates are due to a difference between `minor` and `broad` specifications in the `o_group` column. 

In [126]:
print(nat_desc.loc[nat_desc['Field'] == 'o_group', 'Field Description'].values[0])

SOC occupation level. For most occupations, this field indicates the standard SOC major, minor, broad, and detailed levels, in addition to all-occupations totals. For occupations that OEWS no longer publishes at the SOC detailed level, the “detailed” designation indicates the most detailed data available: either a standard SOC broad occupation or an OEWS-specific combination of detailed occupations. Occupations that OEWS has aggregated to the SOC broad occupation level will appear in the file twice, once with the “broad” and once with the “detailed” designation.


The description of this column does not provide information about how to handle these `minor`/`broad` duplicates. 

They are clearly duplicates, as evidenced by the identical data in nearly all other columns between the matching pairs.

Therefore, at least one duplicate for each needs to be dropped. 

Let's revisit the [SOC User Guide](https://www.bls.gov/soc/socguide.htm):

> The 2000 SOC classifies workers at four levels of aggregation: 1) major group; 2) minor group; 3) broad occupation; and 4) detailed occupation. All occupations are clustered into one of the following 23 major groups:

>  Within these major groups are 96 minor groups, 449 broad occupations, and 821 detailed occupations. Occupations with similar skills or work activities are grouped at each of the four levels of hierarchy to facilitate comparisons. For example, "Life, Physical and Social Science Occupations" (19-0000) is divided into four minor groups, "Life Scientists" (19-1000), "Physical Scientists" (19-2000), "Social Scientists and Related Workers" (19-3000), and "Life, Physical and Social Science Technicians" (19-4000). Life Scientists contains broad occupations such as "Agriculture and Food Scientists" (19-1010), and "Biological Scientists" (19-1020). The broad occupation Biological Scientists includes detailed occupations such as "Biochemists and Biophysicists" (19-1021), and "Microbiologists" (19-1022).

This creates the vague understanding that these row duplicates are the same occupational fields surveyed, but the fields are categorized twice -- as both `minor` and `broad` groups. The purpose for this duplication is unknown.

According to the user guide, the `minor` group only has `96` instances across the whole database. We already have only `94` such groups, as shown previously.

We do know that when BLS adds duplicates, it does so because it has aggregated data from a more `detailed` into a more `broad` direction. The BLS has a track record of making things more general, not more detailed.

Let's continue in this direction by dropping the `broad` rows and keeping the more general `minor` rows.

#### Part 3: Dropping Additional Duplicates

In [127]:
counts_thresh = counts[counts > threshold].index

In [128]:
nat_data_filtered = nat_data_pre_drop[(nat_data_pre_drop['occ_title'].isin(counts_thresh)) & (nat_data_pre_drop['o_group'] == 'broad')]

In [130]:
nat_data_filtered

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
305,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,19-5010,Occupational Health and Safety Specialists and...,...,37.93,48.79,61.13,47790,59620,78900,101490,127140,NaN,NaN
682,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,35-1010,Supervisors of Food Preparation and Serving Wo...,...,21.22,27.4,34.58,29600,35840,44140,56990,71920,NaN,NaN
726,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,37-3010,Grounds Maintenance Workers,...,18.5,22.3,27.14,30140,35470,38470,46390,56460,NaN,NaN
775,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-6010,"Baggage Porters, Bellhops, and Concierges",...,17.7,20.96,26.13,28360,32560,36810,43600,54340,NaN,NaN
779,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,39-7010,Tour and Travel Guides,...,17.63,22.07,28.81,26890,31250,36660,45910,59930,NaN,NaN
817,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,41-4010,"Sales Representatives, Wholesale and Manufactu...",...,35.63,49.61,74.97,38910,50820,74100,103200,155930,NaN,NaN
918,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,43-6010,Secretaries and Administrative Assistants,...,22.82,28.84,36.81,33840,38790,47460,59990,76550,NaN,NaN
1022,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,47-3010,"Helpers, Construction Trades",...,19.44,22.94,27.64,31360,36280,40430,47710,57480,NaN,NaN
1213,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,51-5110,Printing Workers,...,21.55,25.24,30.06,31450,36950,44830,52510,62530,NaN,NaN


In [131]:
nat_data_pre_drop_2 = nat_data_pre_drop.drop(nat_data_filtered.index)

In [132]:
nat_data_pre_drop_2

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,23.8,37.89,60.44,29990,36730,49500,78810,125720,NaN,NaN
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
5,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1020,General and Operations Managers,...,49.5,78.91,#,47420,67160,102950,164130,#,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1396,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7073,Wellhead Pumpers,...,33.66,38.81,46.86,39110,54450,70010,80720,97470,NaN,NaN
1397,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7080,Refuse and Recyclable Material Collectors,...,23.24,29.33,36.16,31810,38330,48350,61010,75200,NaN,NaN
1399,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7120,"Tank Car, Truck, and Ship Loaders",...,27.92,34.25,42.36,38260,47260,58070,71230,88120,NaN,NaN
1401,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-7190,Miscellaneous Material Moving Workers,...,20.04,24.45,31.18,33280,35470,41690,50850,64850,NaN,NaN


Before this data cleaning instance, we had `1396` rows. Our original number of rows that had duplicate data was `516`.

In [133]:
(1396 - 1138) * 2

516

We appear to have dropped the correct amount of rows for this attempt.

In [136]:
nat_data_pre_drop_2[nat_data_pre_drop_2.groupby('occ_title')['occ_title'].transform('count') > threshold]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly


In [137]:
nat_data_pre_drop_2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1138 entries, 0 to 1402
Data columns (total 28 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   area         1138 non-null   int64  
 1   area_title   1138 non-null   object 
 2   area_type    1138 non-null   int64  
 3   prim_state   1138 non-null   object 
 4   naics        1138 non-null   int64  
 5   naics_title  1138 non-null   object 
 6   i_group      1138 non-null   object 
 7   own_code     1138 non-null   int64  
 8   occ_code     1138 non-null   object 
 9   occ_title    1138 non-null   object 
 10  o_group      1138 non-null   object 
 11  tot_emp      1138 non-null   int64  
 12  emp_prse     1138 non-null   float64
 13  h_mean       1138 non-null   object 
 14  a_mean       1138 non-null   object 
 15  mean_prse    1138 non-null   float64
 16  h_pct10      1138 non-null   object 
 17  h_pct25      1138 non-null   object 
 18  h_median     1138 non-null   object 
 19  h_pct75    

In [135]:
nat_data = nat_data_pre_drop_2

#### Remove To Do Item

- ~~Remove duplicate rows discovered via the `occ_title` column.~~

## Hourly and Annual Data Column Cleaning

In [52]:
nat_data['h_pct90'][0]

60.44

In [53]:
print(type(nat_data['h_pct90'][0]))

<class 'float'>


In [54]:
nat_data['h_pct90'][1]

'#'

In [55]:
print(type(nat_data['h_pct90'][1]))

<class 'str'>


## Commentary

Right there, we see that this column has mixed types. For each of the columns in each dataset, we need to resolve this issue. Let's focus on the `h_pct90` column, picked at random.

The first task is to find out how many types there are.

The second task is to find out how many unique types of strings there are.

In [56]:
nat_desc_new.iloc[24:25,:]

,Field,Field Description
24,h_pct90,Hourly 90th percentile wage


In [57]:
nat_desc_new.iloc[:,:]

,Field,Field Description
0,area,"U.S. (99), state FIPS code, Metropolitan Stati..."
1,area_title,Area name
2,area_type,Area type: 1= U.S.; 2= State; 3= U.S. Territor...
3,prim_state,"The primary state for the given area. ""US"" is ..."
4,naics,North American Industry Classification System ...
5,naics_title,North American Industry Classification System ...
6,i_group,Industry level. Indicates cross-industry or NA...
7,own_code,Ownership type: 1= Federal Government; 2= Stat...
8,occ_code,The 6-digit Standard Occupational Classificati...
9,occ_title,SOC title or OEWS-specific title for the occup...


In [58]:
nat_data['h_pct90'].apply(type).unique()

array([<class 'float'>, <class 'str'>, <class 'int'>], dtype=object)

### Commentary

We see `float`, `str`, `int`. 

The float values are probably desirable data points and don't need to be changed.

The `str` values are mysterious. I am not certain what they mean, yet, despite an hour of research on Google and `BLS.gov`. It could mean unreliable data, or data caps, or something else.

The `int` values are also mysterious.

In [59]:
nat_data['h_pct90'][nat_data['h_pct90'].apply(type) == str].unique()

array(['#', '*'], dtype=object)

In [60]:
nat_data['h_pct90'][nat_data['h_pct90'].apply(type) == int].unique()

array([85, 110, 86, 61, 40, 60, 50, 20, 29, 54, 23, 35, 33], dtype=object)

### Commentary

The `str` values can be either `#` or `*`. Again, I don't know what that means. Capped? Unreliable?

The `int` values seem to have a definitive purpose. Perhaps they are code markers.

Found the answer for the `str` values. It was at the bottom of the `Field Description` sheet.

These notes should be a separate dataframe.

In [61]:
nat_desc_new.iloc[36,0]

'#  = indicates a wage equal to or greater than $115.00 per hour or $239,200 per year '

In [62]:
nat_desc_new.iloc[35,0]

'**  = indicates that an employment estimate is not available'

In [63]:
nat_desc_new.iloc[36,0]

'#  = indicates a wage equal to or greater than $115.00 per hour or $239,200 per year '

In [64]:
nat_desc_new.iloc[37,0]

'~ =indicates that the percent of establishments reporting the occupation is less than 0.5%'

### Commentary

We have a conundrum. Starting with the `#` value, this `str` is prevalant in many of the columns. The representation is important and cannot be dropped, and at the same time, it is non-specific.

I think that, what I will do for now, is simply insert the represented lowest value of `115`. That's going to create a skewed dataset. 

I can consider the point at which the given career starts exceeding available ranges. 

Another option is to look at values within the available range, and treat these values that exceed the scope of the dataset as another group. That could be useful -- groups that reach the cap at lower percentiles than others are considered to be more lucrative.

In [65]:
nat_num_only = pd.DataFrame(nat_data['h_pct90'][(nat_data['h_pct90'].apply(type) != str)])

In [66]:
nat_num_only['h_pct90'][nat_num_only['h_pct90'] > 99]

17      104.16
28      105.35
29      105.35
36      105.76
37      105.76
46      102.12
57      105.33
58      105.33
72      109.42
73      109.42
117     102.79
137      111.6
138      111.6
145     100.96
147      99.92
149     101.66
156      99.24
157      99.24
187     107.61
188     107.61
207        110
208        110
232     113.34
264     102.27
265     102.27
269     107.97
339     101.39
342     104.11
449     101.64
474     100.11
476     105.44
509     102.34
537     102.35
538     102.35
810     103.47
811     103.47
1322    101.16
Name: h_pct90, dtype: object

### Commentary

The above results seem to indicate that the values reach `115` at a hard and easily discernable stopping point. Therefore, we should be able to just insert `115` and, when we do later analysis, keep in mind the discrepancy.

The next question: what to do about the `*` symbols? 

In [73]:
nat_data[nat_data['h_pct90'] == '*'].head()

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
7,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1030,Legislators,...,*,*,*,20380,29120,44810,80350,137820,True,NaN
8,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,*,*,20380,29120,44810,80350,137820,True,NaN
45,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-9032,"Education Administrators, Kindergarten through...",...,*,*,*,72400,83840,104070,132550,165820,True,NaN
350,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,25-1000,Postsecondary Teachers,...,*,*,*,47010,61300,81600,125900,180630,True,NaN
351,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,25-1010,"Business Teachers, Postsecondary",...,*,*,*,46460,63040,97270,140360,210530,True,NaN


In [100]:
type(nat_data.loc[nat_data['h_pct90'] == '*','a_pct10'].values[0])

int

### Commentary

An initial inspection shows that the hourly columns are blank when the annual columns have values. 

That raises the question, how many instances exist where hourly is `*` and annual is something other than an int?

In [108]:
nat_data[nat_data['a_pct90'].apply(lambda x: isinstance(x, int)) == False]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN
3,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
4,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
5,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1020,General and Operations Managers,...,49.5,78.91,#,47420,67160,102950,164130,#,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
567,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,29-1249,"Surgeons, All Other",...,#,#,#,77290,139070,#,#,#,NaN,NaN
1317,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-2000,Air Transportation Workers,...,*,*,*,48410,63420,106500,199160,#,True,NaN
1318,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-2010,Aircraft Pilots and Flight Engineers,...,*,*,*,78040,110250,198100,#,#,True,NaN
1319,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,53-2011,"Airline Pilots, Copilots, and Flight Engineers",...,*,*,*,98560,154360,226600,#,#,True,NaN


In [111]:
nat_data[(nat_data['h_pct90'] == '*') & (nat_data['a_pct90'] != '#') & (nat_data['a_pct90'].apply(lambda x: isinstance(x, int)) == False)]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly


### Commentary

A preliminary, quick study shows that if the value in an hourly column is `*`, it is possible that no similar value matches in the associated annual column. However, this search is not thorough enough and needs to be expanded across all hourly and annual columns.

The `h_mean` and `a_mean` columns are likely composites of the two different types of numbers.

Logic stands to reason that we can simplify our search to see wehther these are ever both equal to `*` at the same time. If not, then we can assume, for now, that the two types of columns do not conflict.

In [113]:
nat_data[(nat_data['h_mean'] == '*') & (nat_data['a_mean'] == '*')]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly


### Commentary

There appears to be no overlap between the two columns.

The next question is whether there are rows where both `hourly` and `annual` values are provided.

In [114]:
nat_data[(nat_data['h_mean'].apply(lambda x: isinstance(x, int)) == True) & (nat_data['a_mean'].apply(lambda x: isinstance(x, int)) == True)]

,area,area_title,area_type,prim_state,naics,naics_title,i_group,own_code,occ_code,occ_title,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
5,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1020,General and Operations Managers,...,49.5,78.91,#,47420,67160,102950,164130,#,NaN,NaN
6,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,49.5,78.91,#,47420,67160,102950,164130,#,NaN,NaN
189,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,17-2070,Electrical and Electronics Engineers,...,57.11,72.37,87.21,76190,92390,118780,150530,181390,NaN,NaN
346,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,23-2090,Miscellaneous Legal Support Workers,...,29.11,39.13,62.75,38170,46950,60550,81390,130510,NaN,NaN
791,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,41-0000,Sales and Related Occupations,...,18.01,28.79,47.53,27060,31010,37460,59880,98860,NaN,NaN
920,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,43-6012,Legal Secretaries and Administrative Assistants,...,26.03,34.66,42.15,35530,42720,54140,72090,87660,NaN,NaN
924,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,43-9020,Data Entry and Information Processing Workers,...,19.94,23.7,28.47,31200,35760,41480,49300,59220,NaN,NaN
950,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,45-2020,Animal Breeders,...,25,28.67,43.26,37130,43310,52000,59630,89970,NaN,NaN
951,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,45-2021,Animal Breeders,...,25,28.67,43.26,37130,43310,52000,59630,89970,NaN,NaN
1121,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,49-9044,Millwrights,...,31.33,38.74,45.07,45100,55420,65170,80580,93740,NaN,NaN


The answer appears to be, yes. There are `13` rows where both annual and hourly data is provided.

More information will be needed to understand why these industries' parameters allow for both types of pay.

### To Do List - National Data

#### Data

- Drop the four columns that have all blanks
- Keep in mind issue where 13 rows have both annual and hourly data
- In all hourly data rows, insert `115` for all `#`
- In all annual data rows, insert `239,200` for all `#`
- In all annual and hourly rows, insert `null` for all `*`

#### Descriptions

- Separate the notes at the end into a new database (may only need one for everything
- Drop the columns to match the dropped columns in the data 